# Импорты и загрузка таблиц

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import  deepcopy
from torch.utils.data import DataLoader, TensorDataset
from datetime import datetime

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / "data" 
RATING_PATH = DATASET_PATH / "raw" / "ratings.csv"

ratings_df = pd.read_csv(RATING_PATH, encoding="latin-1")

ratings_df["timestamp"] = pd.to_datetime(ratings_df["timestamp"], unit="s")

In [5]:
MOVIES_PATH = DATASET_PATH / "raw" / "movies.csv"

movies_df = pd.read_csv(MOVIES_PATH, encoding="latin-1")

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
TRAIN_PATH = DATASET_PATH / "processed" / "train_20261020_161901.csv"
TEST_PATH = DATASET_PATH / "processed" / "test_20261020_161901.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# Подготовка индексов пользователей и фильмов

In [7]:
# Уникальные пользователи и фильмы, известные на обучении
user_ids = np.sort(train["userId"].unique())
movie_ids = np.sort(train["movieId"].unique())

# Перевод исходных ID в индексы от 0 до N-1
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

# Создаём копии, чтобы не изменять исходные таблицы
train_model = train.copy()
test_model = test.copy()

train_model["user_idx"] = train_model["userId"].map(user_to_idx)
train_model["movie_idx"] = train_model["movieId"].map(movie_to_idx)

test_model["user_idx"] = test_model["userId"].map(user_to_idx)
test_model["movie_idx"] = test_model["movieId"].map(movie_to_idx)

# Оставляем в тесте только пары, для которых известны оба эмбеддинга
test_known = test_model.dropna(subset=["user_idx", "movie_idx"]).copy()

train_model["user_idx"] = train_model["user_idx"].astype("int64")
train_model["movie_idx"] = train_model["movie_idx"].astype("int64")
test_known["user_idx"] = test_known["user_idx"].astype("int64")
test_known["movie_idx"] = test_known["movie_idx"].astype("int64")

# Разбиение обучающей части

In [8]:
fit_parts = []
val_parts = []

for _, group in train_model.groupby("user_idx", sort=False):
    group = group.sort_values("timestamp")
    split_idx = int(len(group) * 0.9)
    
    if len(group) >= 3:
        fit_parts.append(group.iloc[:split_idx])
        val_parts.append(group.iloc[split_idx:])
    else:
        fit_parts.append(group)
        
fit_df = pd.concat(fit_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)

In [9]:
print("Fit:", fit_df.shape)
print("Validation:", val_df.shape)

print("Пользователей в fit:", fit_df["user_idx"].nunique())
print("Пользователей в validation:", val_df["user_idx"].nunique())

print(
    "Пересечение строк по пользователям:",
    len(set(fit_df.index) & set(val_df.index))
)

Fit: (72115, 6)
Validation: (8304, 6)
Пользователей в fit: 610
Пользователей в validation: 610
Пересечение строк по пользователям: 8304


# Подготовка жанровой матрицы

In [9]:
genres_by_movie = (movies_df.drop_duplicates("movieId").set_index("movieId")["genres"].to_dict())

genre_sets = {}

for movie_id in movie_ids:
    genres = genres_by_movie.get(movie_id, "")
    
    if pd.isna(genres) or genres == ("(no genres listed)"):
        genre_sets[movie_id] = set()
    else:
        genre_sets[movie_id] = set(genres.split("|"))
        
genre_names = sorted({genre for genres in genre_sets.values() for genre in genres})

genre_to_idx = {genre: idx for idx, genre in enumerate(genre_names)}

genre_matrix = np.zeros((len(movie_ids), len(genre_names)), dtype=np.float32)

for movie_id, genres in genre_sets.items():
    movie_idx = movie_to_idx[movie_id]
    
    for genre in genres:
        genre_matrix[movie_idx, genre_to_idx[genre]] = 1.0
        
print("Фильмов:", genre_matrix.shape[0])
print("Жанров:", genre_matrix.shape[1])
print("Размер матрицы:", genre_matrix.shape)
print("Жанры:", genre_names)

Фильмов: 8230
Жанров: 19
Размер матрицы: (8230, 19)
Жанры: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [10]:
assert genre_matrix.shape[0] == len(movie_ids)
assert genre_matrix.shape[1] == len(genre_names)
assert genre_matrix.dtype == np.float32

assert np.isfinite(genre_matrix).all()
assert set(genre_to_idx) == set(genre_names)

print("Проверки жанровой матрицы пройдены.")

Проверки жанровой матрицы пройдены.


# Обучение жанровой bpr-модели

In [10]:
fit_pos = (
    fit_df.loc[fit_df["rating"] >= 4, ["user_idx", "movie_idx"]]
    .copy()
    .reset_index(drop=True)
)

fit_pos["user_idx"] = fit_pos["user_idx"].astype("int64")
fit_pos["movie_idx"] = fit_pos["movie_idx"].astype("int64")

n_users = len(user_ids)
n_movies = len(movie_ids)

all_movie_idx = np.arange(n_movies, dtype=np.int64)

pos_movies_by_user = (fit_pos.groupby("user_idx")["movie_idx"].apply(set).to_dict())

neg_candidates_by_user = {}

for user_idx, pos_movies in pos_movies_by_user.items():
    neg_candidates_by_user[user_idx] = np.setdiff1d(
        all_movie_idx,
        np.array(list(pos_movies), dtype=np.int64)
    )
    
rng = np.random.default_rng(42)

bpr_users = fit_pos["user_idx"].to_numpy(dtype=np.int64)
bpr_pos = fit_pos["movie_idx"].to_numpy(dtype=np.int64)
bpr_neg = np.empty(len(fit_pos), dtype=np.int64)

for i, user_idx in enumerate(bpr_users):
    candidates = neg_candidates_by_user[user_idx]
    bpr_neg[i] = rng.choice(candidates)
    
genre_features = genre_matrix / np.maximum(genre_matrix.sum(axis=1, keepdims=True), 1.0)

val_pos_df = val_df.loc[val_df["rating"] >= 4].copy()

print("Пользователей:", n_users)
print("Фильмов:", n_movies)
print("Положительных обучающих оценок:", len(fit_pos))
print("Обучающих троек BPR:", len(bpr_users))
print("Положительных оценок в валидации:", len(val_pos_df))
print("Размерность жанровых признаков:", genre_features.shape)

Пользователей: 610
Фильмов: 8230
Положительных обучающих оценок: 35229
Обучающих троек BPR: 35229
Положительных оценок в валидации: 3842
Размерность жанровых признаков: (8230, 19)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

class GenreBPR(nn.Module):
    def __init__(self, n_users, n_movies, genre_features, emb_dim=32):
        super().__init__()
        
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        
        self.genre_projection = nn.Linear(genre_features.shape[1], emb_dim, bias=False)
        
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        self.register_buffer(
            "genre_features", 
            torch.as_tensor(genre_features, dtype=torch.float32)
        )
        
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.movie_emb.weight, std=0.01)
        nn.init.normal_(self.genre_projection.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
        
        self.genre_weight = nn.Parameter(torch.tensor(0.1))
        
    def score(self, user_idx, movie_idx):
        user_vec = self.user_emb(user_idx)
        movie_vec = self.movie_emb(movie_idx)
        
        genre_vec = self.genre_projection(self.genre_features[movie_idx])
        
        movie_vec = movie_vec + genre_vec * self.genre_weight
        
        interaction = (user_vec * movie_vec).sum(dim=1)
        
        user_bias = self.user_bias(user_idx).squeeze(-1)
        movie_bias = self.movie_bias(movie_idx).squeeze(-1)
        
        return interaction + user_bias + movie_bias

Устройство: cuda


In [12]:
def train_genre_bpr(
    model, 
    users, 
    pos_movies, 
    neg_movies,
    fit_df,
    val_pos_df,
    epochs=20, 
    batch_size=1024,
    lr=0.005,
    weight_decay=1e-4,
    patience=3
):
    dataset = TensorDataset(
        torch.as_tensor(users, dtype=torch.long),
        torch.as_tensor(pos_movies, dtype=torch.long),
        torch.as_tensor(neg_movies, dtype=torch.long)
    )
    
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = []
    best_ndcg = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0
        
        for batch_users, pos_movies, neg_movies in loader:
            batch_users = batch_users.to(device)
            pos_movies = pos_movies.to(device)
            neg_movies = neg_movies.to(device)
            
            optimizer.zero_grad()
            
            pos_scores = model.score(batch_users, pos_movies)
            neg_scores = model.score(batch_users, neg_movies)
            
            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * len(batch_users)
            total_examples += len(batch_users)
        
        epoch_loss = total_loss / total_examples
        
        val_metrics = evaluate_ranking(
            model=model,
            fit_df=fit_df,
            val_pos_df=val_pos_df,
            k=10
        )
        val_ndcg = val_metrics["NDCG@10"]
        val_recall = val_metrics["Recall@10"]
        
        history.append({
            "epoch": epoch + 1,
            "train_loss": epoch_loss,
            "val_recall@10": val_recall,
            "val_ndcg@10": val_ndcg,
        })
        
        print(
            f"Эпоха {epoch + 1:02d}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Recall@10: {val_recall:.4f} | "
            f"NDCG@10: {val_ndcg:.4f}"
        )
        
        if val_ndcg > best_ndcg:
            best_ndcg = val_ndcg
            best_epoch = epoch + 1
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            
        if epochs_without_improvement >= patience:
            print(f"Ранняя остановка: эпоха {epoch + 1}.")
            break
            
    model.load_state_dict(best_state)
    
    print(f"\nЛучшая эпоха: {best_epoch}")
    print(f"Лучший Val NDCG@10: {best_ndcg:.4f}")
    
    return model, pd.DataFrame(history)

In [13]:
sample_users = torch.tensor(bpr_users[:4], dtype=torch.long, device=device)
sample_movies = torch.tensor(bpr_pos[:4], dtype=torch.long, device=device)

In [ ]:
genre_model_test = GenreBPR(
    n_users=n_users,
    n_movies=n_movies,
    genre_features=genre_features,
    emb_dim=32
).to(device)

sample_users = torch.tensor(bpr_users[:4], dtype=torch.long, device=device)
sample_movies = torch.tensor(bpr_pos[:4], dtype=torch.long, device=device)

with torch.no_grad():
    sample_scores = genre_model_test.score(sample_users, sample_movies)

print("Матрица жанров:", genre_model_test.genre_features.shape)
print("Размер эмбеддинга пользователя:", genre_model_test.user_emb.weight.shape)
print("Размер эмбеддинга фильма:", genre_model_test.movie_emb.weight.shape)
print("Оценки четырёх пар:", sample_scores.shape)
print("Пример оценок:", sample_scores.cpu().numpy())

In [28]:
torch.manual_seed(42)
np.random.seed(42)

genre_model = GenreBPR(
    n_users=n_users,
    n_movies=n_movies,
    genre_features=genre_features,
    emb_dim=32
).to(device)

genre_model, genre_history = train_genre_bpr(
    model=genre_model,
    users=bpr_users,
    pos_movies=bpr_pos,
    neg_movies=bpr_neg,
    fit_df=fit_df,
    val_pos_df=val_pos_df,
    epochs=20,
    batch_size=1024,
    lr=0.005,
    weight_decay=1e-4,
    patience=3
)

print("Итоговый вес жанров:", genre_model.genre_weight.item())

Эпоха 01/20 | Loss: 0.6638 | Recall@10: 0.0403 | NDCG@10: 0.0296
Эпоха 02/20 | Loss: 0.4761 | Recall@10: 0.0429 | NDCG@10: 0.0305
Эпоха 03/20 | Loss: 0.3007 | Recall@10: 0.0536 | NDCG@10: 0.0385
Эпоха 04/20 | Loss: 0.2463 | Recall@10: 0.0500 | NDCG@10: 0.0365
Эпоха 05/20 | Loss: 0.2253 | Recall@10: 0.0481 | NDCG@10: 0.0351
Эпоха 06/20 | Loss: 0.2123 | Recall@10: 0.0508 | NDCG@10: 0.0345
Ранняя остановка: эпоха 6.

Лучшая эпоха: 3
Лучший Val NDCG@10: 0.0385
Итоговый вес жанров: 0.5456116795539856


In [15]:
def evaluate_ranking(model, fit_df, val_pos_df, k=10):
    model.eval()
    seen_by_user = (fit_df.groupby("user_idx")["movie_idx"].apply(set).to_dict())
    
    relevant_by_user = (val_pos_df.groupby("user_idx")["movie_idx"].apply(set).to_dict())
    
    recalls = []
    ndcgs = []
    
    all_movies = np.arange(n_movies, dtype=np.int64)
    
    with torch.no_grad():
        for user_idx, relevant_movies in relevant_by_user.items():
            seen_movies = seen_by_user.get(user_idx, set())
            relevant_movies = relevant_movies - seen_movies
            
            if not relevant_movies:
                continue
                
            candidates = np.array(
                [m for m in all_movies if m not in sample_movies], 
                dtype=np.int64
            )
            
            user_tensor = torch.full(
                (len(candidates),), 
                user_idx, 
                dtype=torch.long, 
                device=device
            )
            
            movie_tensor = torch.as_tensor(candidates.copy(), dtype=torch.long, device=device)
            
            scores = model.score(user_tensor, movie_tensor)
            top_indices = torch.topk(scores, k=min(k, len(candidates))).indices.cpu().numpy()
            
            top_movies = candidates[top_indices]
            hits = np.array([int(movie in relevant_movies) for movie in top_movies])
            
            recall = hits.sum() / len(relevant_movies)
            
            discounts = 1 / np.log2(np.arange(2, len(hits) + 2))
            dcg = (hits * discounts).sum()
            
            ideal_hits = min(len(relevant_movies), k)
            ideal_discounts = 1 / np.log2(np.arange(2, ideal_hits + 2))
            idcg = ideal_discounts.sum()
            
            recalls.append(recall)
            ndcgs.append(dcg / idcg)
            
    return {
        f"Recall@{k}": float(np.mean(recalls)),
        f"NDCG@{k}": float(np.mean(ndcgs)),
        "users_evaluated": len(recalls),
    }

In [25]:
genre_val_metrics = evaluate_ranking(
    model=genre_model,
    fit_df=fit_df,
    val_pos_df=val_pos_df,
    k=10
)

print("Жанровая BPR — validation:")
for metric, value in genre_val_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

Жанровая BPR — validation:
Recall@10: 0.0446
NDCG@10: 0.0315
users_evaluated: 555


In [31]:
BASE_CONFIG = {
    "genre_weight": 0.1, 
    "embedding_dim": 32, 
    "lr": 0.005, 
    "weight_decay": 1e-4
}

experiments = []

for v in [0.0, 0.1, 0.25, 0.5, 1.0]:
    config = BASE_CONFIG.copy()
    config["genre_weight"] = v
    experiments.append({"parameter": "genre_weight", **config})
    
for v in [8, 16, 64]:
    config = BASE_CONFIG.copy()
    config["embedding_dim"] = v
    experiments.append({"parameter": "embedding_dim", **config})

for v in [0.001, 0.003, 0.01]:
    config = BASE_CONFIG.copy()
    config["lr"] = v
    experiments.append({"parameter": "lr", **config})

for v in [1e-5, 1e-3, 1e-2]:
    config = BASE_CONFIG.copy()
    config["weight_decay"] = v
    experiments.append({"parameter": "weight_decay", **config})

experiments_df = pd.DataFrame(experiments)

print("Количество экспериментов:", len(experiments_df))
display(experiments_df)

Количество экспериментов: 14


,parameter,genre_weight,embedding_dim,lr,weight_decay
0,genre_weight,0.00,32,0.005,0.00010
1,genre_weight,0.10,32,0.005,0.00010
2,genre_weight,0.25,32,0.005,0.00010
3,genre_weight,0.50,32,0.005,0.00010
4,genre_weight,1.00,32,0.005,0.00010
5,embedding_dim,0.10,8,0.005,0.00010
6,embedding_dim,0.10,16,0.005,0.00010
7,embedding_dim,0.10,64,0.005,0.00010
8,lr,0.10,32,0.001,0.00010
9,lr,0.10,32,0.003,0.00010


In [32]:
results = []
all_histories = []
    
torch.manual_seed(42)
np.random.seed(42)

model = GenreBPR(
    n_users=n_users,
    n_movies=n_movies,
    genre_features=genre_features,
    emb_dim=config["embedding_dim"]
).to(device)

if isinstance(model.genre_weight, torch.nn.Parameter):
    with torch.no_grad():
        model.genre_weight.fill_(config["genre_weight"])
    model.genre_weight.requires_grad_(False)
else:
    model.genre_weight = float(config["genre_weight"])

model, history = train_genre_bpr(
    model=model,
    users=bpr_users,
    pos_movies=bpr_pos,
    neg_movies=bpr_neg,
    fit_df=fit_df,
    val_pos_df=val_pos_df,
    epochs=20,
    batch_size=1024,
    lr=config["lr"],
    weight_decay=config["weight_decay"],
    patience=3,
)

Эпоха 01/20 | Loss: 0.6643 | Recall@10: 0.0479 | NDCG@10: 0.0331
Эпоха 02/20 | Loss: 0.4632 | Recall@10: 0.0622 | NDCG@10: 0.0401
Эпоха 03/20 | Loss: 0.2404 | Recall@10: 0.0602 | NDCG@10: 0.0421
Эпоха 04/20 | Loss: 0.1600 | Recall@10: 0.0574 | NDCG@10: 0.0401
Эпоха 05/20 | Loss: 0.1180 | Recall@10: 0.0614 | NDCG@10: 0.0421
Эпоха 06/20 | Loss: 0.0902 | Recall@10: 0.0589 | NDCG@10: 0.0417
Ранняя остановка: эпоха 6.

Лучшая эпоха: 3
Лучший Val NDCG@10: 0.0421


In [12]:
RES_DIR = PROJECT_ROOT / "reports"
RES_DIR.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

RES_PATH = RES_DIR / f"genre_bpr_sweep_{run_id}.csv"
HISTORY_PATH = RES_DIR / f"genre_bpr_history_{run_id}.csv"

# Таблица с итогами каждого запуска
RESULT_COLUMNS = [
    "experiment_id",
    "parameter",
    "genre_weight",
    "embedding_dim",
    "lr",
    "weight_decay",
    "best_epoch",
    "best_train_loss",
    "val_recall@10",
    "val_ndcg@10",
]

# История метрик по эпохам
HISTORY_COLUMNS = [
    "experiment_id",
    "parameter",
    "genre_weight",
    "embedding_dim",
    "lr",
    "weight_decay",
    "epoch",
    "train_loss",
    "val_recall@10",
    "val_ndcg@10",
]

pd.DataFrame(columns=RESULT_COLUMNS).to_csv(RES_PATH, index=False)
pd.DataFrame(columns=HISTORY_COLUMNS).to_csv(HISTORY_PATH, index=False)

# Номер следующего эксперимента
experiment_id = 1

print("Файлы созданы:")
print("Результаты:", RES_PATH)
print("История обучения:", HISTORY_PATH)

Файлы созданы:
Результаты: C:\IT\movie-recommender-pytorch\reports\genre_bpr_sweep_20261026_104605.csv
История обучения: C:\IT\movie-recommender-pytorch\reports\genre_bpr_history_20261026_104605.csv


In [31]:
config = {
    "parameter": "lr",
    "genre_weight": 0.1,
    "embedding_dim": 32,
    "lr": 0.005,
    "weight_decay": 1e-5,
}
print(experiment_id)

12


In [13]:
# Таблица с итогами каждого запуска
RESULT_COLUMNS = [
    "experiment_id",
    "parameter",
    "genre_weight",
    "embedding_dim",
    "lr",
    "weight_decay",
    "best_epoch",
    "best_train_loss",
    "val_recall@10",
    "val_ndcg@10",
]

# История метрик по эпохам
HISTORY_COLUMNS = [
    "experiment_id",
    "parameter",
    "genre_weight",
    "embedding_dim",
    "lr",
    "weight_decay",
    "epoch",
    "train_loss",
    "val_recall@10",
    "val_ndcg@10",
]

In [3]:
RES_DIR = PROJECT_ROOT / "reports"

RES_PATH = RES_DIR / f"genre_bpr_sweep_20261026_104605.csv"
HISTORY_PATH = RES_DIR / f"genre_bpr_history_20261026_104605.csv"

In [15]:
experiment_id = 7

In [33]:
# Находим эпоху с лучшим NDCG@10
best_idx = history["val_ndcg@10"].idxmax()
best_row = history.loc[best_idx]

result = {
    "experiment_id": experiment_id,
    **config,
    "best_epoch": int(best_row["epoch"]),
    "best_train_loss": float(best_row["train_loss"]),
    "val_recall@10": float(best_row["val_recall@10"]),
    "val_ndcg@10": float(best_row["val_ndcg@10"]),
}

# Добавляем итоговую строку, не перезаписывая файл
pd.DataFrame([result]).to_csv(
    RES_PATH,
    mode="a",
    header=False,
    index=False,
)

# Добавляем историю эпох
history_copy = history.copy()
history_copy["experiment_id"] = experiment_id

for name in ["parameter", "genre_weight", "embedding_dim", "lr", "weight_decay"]:
    history_copy[name] = config[name]

history_copy = history_copy[HISTORY_COLUMNS]

history_copy.to_csv(
    HISTORY_PATH,
    mode="a",
    header=False,
    index=False,
)

print("Эксперимент сохранён:", experiment_id)
print(f"Best epoch: {result['best_epoch']}")
print(f"Recall@10: {result['val_recall@10']:.4f}")
print(f"NDCG@10: {result['val_ndcg@10']:.4f}")

experiment_id += 1

Эксперимент сохранён: 12
Best epoch: 3
Recall@10: 0.0602
NDCG@10: 0.0421


# Сравнение с бейзлайном и обычным bpr

In [4]:
res_df = pd.read_csv(RES_PATH)
display(res_df)

,experiment_id,parameter,genre_weight,embedding_dim,lr,weight_decay,best_epoch,best_train_loss,val_recall@10,val_ndcg@10
0,1,genre_weight,0.00,32,0.005,0.00010,2,0.489052,0.057592,0.042008
1,2,genre_weight,0.10,32,0.005,0.00010,3,0.302962,0.057018,0.041993
2,3,genre_weight,0.25,32,0.005,0.00010,4,0.248861,0.051621,0.040632
3,4,genre_weight,0.50,32,0.005,0.00010,4,0.256015,0.050481,0.037173
4,5,genre_weight,1.00,32,0.005,0.00010,5,0.241001,0.047728,0.032950
5,6,embedding_dim,0.10,16,0.005,0.00010,8,0.223838,0.057803,0.042609
6,6,embedding_dim,0.10,16,0.005,0.00010,8,0.223838,0.057803,0.042609
7,7,lr,0.10,32,0.001,0.00010,1,0.688550,0.053319,0.035089
8,8,lr,0.10,32,0.003,0.00010,4,0.309051,0.051244,0.039377
9,9,lr,0.10,32,0.010,0.00010,4,0.229473,0.060521,0.043471


In [16]:
best_config = {
    "embedding_dim": 32,
    "genre_weight": 0.1,
    "lr": 0.01,
    "weight_decay": 1e-4,
}

genre_bpr_test_model = GenreBPR(
    n_users=n_users,
    n_movies=n_movies,
    genre_features=genre_features,
    emb_dim=best_config["embedding_dim"]
).to(device)

if isinstance(genre_bpr_test_model.genre_weight, torch.nn.Parameter):
    with torch.no_grad():
        genre_bpr_test_model.genre_weight.fill_(best_config["genre_weight"])
    genre_bpr_test_model.genre_weight.requires_grad_(False)
else:
    genre_bpr_test_model.genre_weight = best_config["genre_weight"]
    
genre_bpr_test_model, genre_bpr_history = train_genre_bpr(
    model=genre_bpr_test_model,
    users=bpr_users,
    pos_movies=bpr_pos,
    neg_movies=bpr_neg,
    fit_df=fit_df,
    val_pos_df=val_pos_df,
    epochs=20,
    batch_size=1024,
    lr=best_config["lr"],
    weight_decay=best_config["weight_decay"],
    patience=3
)

Эпоха 01/20 | Loss: 0.6203 | Recall@10: 0.0474 | NDCG@10: 0.0339
Эпоха 02/20 | Loss: 0.3329 | Recall@10: 0.0550 | NDCG@10: 0.0398
Эпоха 03/20 | Loss: 0.2594 | Recall@10: 0.0595 | NDCG@10: 0.0418
Эпоха 04/20 | Loss: 0.2379 | Recall@10: 0.0543 | NDCG@10: 0.0409
Эпоха 05/20 | Loss: 0.2209 | Recall@10: 0.0611 | NDCG@10: 0.0429
Эпоха 06/20 | Loss: 0.2092 | Recall@10: 0.0617 | NDCG@10: 0.0433
Эпоха 07/20 | Loss: 0.2011 | Recall@10: 0.0626 | NDCG@10: 0.0431
Эпоха 08/20 | Loss: 0.1952 | Recall@10: 0.0537 | NDCG@10: 0.0401
Эпоха 09/20 | Loss: 0.1913 | Recall@10: 0.0606 | NDCG@10: 0.0428
Ранняя остановка: эпоха 9.

Лучшая эпоха: 6
Лучший Val NDCG@10: 0.0433


In [17]:
test_pos_df = test_known.loc[test_known["rating"] >= 4].copy()

test_metrics = evaluate_ranking(
    model=genre_bpr_test_model,
    fit_df=train_model,
    val_pos_df=test_pos_df,
    k=10,
)

print("GenreBPR — тестовые метрики")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, (int, float)) else f"{metric}: {value}")

GenreBPR — тестовые метрики
Recall@10: 0.0352
NDCG@10: 0.0418
users_evaluated: 591.0000


# Финальная оценка выбранной модели на тесте

In [18]:
def generate_bpr_pairs(pos_df, pos_movies_by_user, all_movie_indices):
    rng = np.random.default_rng(42)

    users = pos_df["user_idx"].to_numpy()
    pos_movies = pos_df["movie_idx"].to_numpy()

    neg_movies = np.empty(len(pos_df), dtype=np.int64)

    for i, user_idx in enumerate(users):
        pos_set = pos_movies_by_user[user_idx]

        while True:
            neg_movie = rng.choice(all_movie_indices)

            if neg_movie not in pos_set:
                neg_movies[i] = neg_movie
                break

    return users, pos_movies, neg_movies

In [20]:
pos_train = train_model.loc[train_model["rating"] >= 4].copy()

pos_movies_by_user = (pos_train.groupby("user_idx")["movie_idx"].apply(set).to_dict())

all_movie_indices = np.arange(n_movies)

final_users, final_pos, final_neg = generate_bpr_pairs(
    pos_train, 
    pos_movies_by_user, 
    all_movie_indices
)

print("Положительных взаимодействий:", len(pos_train))
print("Обучающих троек:", len(final_users))
print("Пользователей:", train_model["user_idx"].nunique())
print("Фильмов:", n_movies)

Положительных взаимодействий: 39071
Обучающих троек: 39071
Пользователей: 610
Фильмов: 8230


In [24]:
def train_bpr_fixed_epochs(
    model, 
    users, 
    pos_movies, 
    neg_movies, 
    epochs, 
    lr, 
    weight_decay, 
    batch_size=1024
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    users_t = torch.as_tensor(users, dtype=torch.long, device=device)
    pos_t = torch.as_tensor(pos_movies, dtype=torch.long, device=device)
    neg_t = torch.as_tensor(neg_movies, dtype=torch.long, device=device)
    
    history = []
    
    model.train()
    
    for epoch in range(epochs):
        permutation = torch.randperm(len(users_t), device=device)
        epoch_loss = 0.0
        n_samples = 0
        
        for start in range(0, len(users_t), batch_size):
            idx = permutation[start:start + batch_size]
            
            pos_scores = model.score(users_t[idx], pos_t[idx])
            neg_scores = model.score(users_t[idx], neg_t[idx])
            
            loss = - F.logsigmoid(pos_scores - neg_scores).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            batch_n = len(idx)
            epoch_loss += loss.item() * batch_n
            n_samples += batch_n
            
        mean_loss = epoch_loss / n_samples
        history.append({
            "epoch": epoch + 1,
            "train_loss": mean_loss,
        })
        
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"| loss={mean_loss:.4f}"
        )
    
    return model, history

In [25]:
final_genre_bpr = GenreBPR(
    n_users=n_users,
    n_movies=n_movies,
    genre_features=genre_features,
    emb_dim=32
).to(device)

if isinstance(final_genre_bpr.genre_weight, torch.nn.Parameter):
    with torch.no_grad():
        final_genre_bpr.genre_weight.fill_(0.1)
    final_genre_bpr.genre_weight.requires_grad_(False)
else:
    final_genre_bpr.genre_weight = 0.1
    
final_genre_bpr, final_genre_history = train_bpr_fixed_epochs(
    model=final_genre_bpr,
    users=final_users,
    pos_movies=final_pos,
    neg_movies=final_neg,
    epochs=4,
    lr=0.01,
    weight_decay=1e-4,
)

Epoch 1/4 | loss=0.6083
Epoch 2/4 | loss=0.3256
Epoch 3/4 | loss=0.2655
Epoch 4/4 | loss=0.2461


In [27]:
final_genre_test_metrics = evaluate_ranking(
    model=final_genre_bpr,
    fit_df=train_model,
    val_pos_df=test_known[test_known["rating"] >= 4],
    k=10,
)

print("GenreBPR — тестовые метрики")
for metric, value in final_genre_test_metrics.items():
    print(f"{metric}: {value:.4f}")

GenreBPR — тестовые метрики
Recall@10: 0.0395
NDCG@10: 0.0467
users_evaluated: 591.0000


# Тестирование скриптов

In [1]:
from src.data import (
    load_split_data,
    prepare_model_data,
    chronological_validation_split
)

train, test = load_split_data()
data = prepare_model_data(train, test)

fit_df, val_df, val_pos_df = chronological_validation_split(data["train_model"])

print("Train:", len(data["train_model"]))
print("Test known:", len(data["test_known"]))
print("Users:", len(data["user_ids"]))
print("Movies:", len(data["movie_ids"]))
print("Fit:", len(fit_df))
print("Validation:", len(val_df))
print("Positive validation:", len(val_pos_df))

Train: 80419
Test known: 18715
Users: 610
Movies: 8230
Fit: 72115
Validation: 8304
Positive validation: 3842


In [2]:
import torch
from src.model import MatrixFactorization

model = MatrixFactorization(
    n_users=610,
    n_movies=8230,
    emb_dim=16,
)

users = torch.tensor([0, 1, 2])
movies = torch.tensor([10, 20, 30])

scores = model.score(users, movies)

print("Scores:", scores)
print("Output shape:", scores.shape)

Scores: tensor([3.4857, 3.4953, 3.4938], grad_fn=<AddBackward0>)
Output shape: torch.Size([3])
